# اختبارات الصمود لنتيجة Vmem — الورم الدبقي (GBM)

النتيجة الأولى: كل +1 mV في Vmem المتوقع ⟵ خطر وفاة أعلى ~8% (p = 0.007).
**هذا الدفتر يسأل: هل النتيجة حقيقية أم وهم؟** ويجيب بأربعة اختبارات:

| الاختبار | السؤال |
|---|---|
| A | هل تبقى العلاقة بعد ضبط **العمر**؟ |
| B | هل هي مجرد تأثير **جين واحد** (مثل SCN9A)؟ |
| C | هل تعطي **جينات عشوائية** نتيجة مماثلة؟ (الأهم) |
| D | الخلاصة بكلمات بسيطة |

**التشغيل:** Runtime ← Run all. يستغرق 3–6 دقائق (جلب جينات عشوائية + 1000 محاكاة).

In [ ]:
!pip install lifelines requests pandas matplotlib -q
print("تم التثبيت")

## 1) المحرك الفيزيائي (نفس السابق + نسخة سريعة للمصفوفات)

In [ ]:
import numpy as np, pandas as pd

R, F, T_BODY = 8.314, 96485, 310.15
CONC = dict(K_out=5.0, K_in=140.0, Na_out=145.0, Na_in=15.0, Cl_in=10.0, Cl_out=110.0)
BASELINE = dict(p_k=1.0, p_na=0.02, p_cl=0.10)
RTF = (R * T_BODY) / F * 1000.0

def ghk_vmem(p_k, p_na, p_cl):
    num = p_k * CONC["K_out"] + p_na * CONC["Na_out"] + p_cl * CONC["Cl_in"]
    den = p_k * CONC["K_in"] + p_na * CONC["Na_in"] + p_cl * CONC["Cl_out"]
    return RTF * np.log(num / den)

def scale_matrix(expr, reference, max_scale=5.0):
    """نفس منطق log2 السابق لكن لكل المرضى دفعة واحدة. القيم الناقصة = 1."""
    s = np.log2(np.clip(expr, 1e-6, None) + 1) / np.log2(reference + 1)
    s = np.clip(s, 1.0 / max_scale, max_scale)
    return np.where(np.isnan(expr), 1.0, s)

def vmem_from_roles(expr_df, roles):
    """
    expr_df: صفوف = مرضى، أعمدة = جينات
    roles: {"K": [جينات], "Na": [جينات], "Cl": [جينات]}
    """
    ref = expr_df.mean(axis=0)
    agg = {}
    for ch, genes in roles.items():
        if genes:
            m = scale_matrix(expr_df[genes].to_numpy(dtype=float), ref[genes].to_numpy(dtype=float))
            agg[ch] = m.mean(axis=1)
        else:
            agg[ch] = np.ones(len(expr_df))
    v = ghk_vmem(BASELINE["p_k"] * agg["K"], BASELINE["p_na"] * agg["Na"], BASELINE["p_cl"] * agg["Cl"])
    return pd.Series(v, index=expr_df.index, name="vmem")

print("خط الأساس:", round(float(ghk_vmem(1.0, 0.02, 0.10)), 2), "mV (المتوقع ≈ -75.7)")

## 2) جلب البيانات: التعبير + البقاء + العمر + جينات عشوائية للمقارنة

In [ ]:
import requests

API = "https://www.cbioportal.org/api"
STUDY = "gbm_tcga_pan_can_atlas_2018"
SAMPLE_LIST = f"{STUDY}_rna_seq_v2_mrna"
PROFILE = f"{STUDY}_rna_seq_v2_mrna"

ROLES = {"Na": ["SCN9A"], "K": ["KCNJ10", "KCNJ2"], "Cl": ["CLIC1"]}   # نفس الجينات السابقة
OUR_GENES = [g for gs in ROLES.values() for g in gs]

def entrez_map(symbols):
    r = requests.post(f"{API}/genes/fetch", params={"geneIdType": "HUGO_GENE_SYMBOL"},
                      json=symbols, timeout=30)
    r.raise_for_status()
    return {g["hugoGeneSymbol"]: g["entrezGeneId"] for g in r.json()}

def fetch_molecular(entrez_ids):
    """يرجع DataFrame: صفوف = sampleId، أعمدة = entrezGeneId."""
    r = requests.post(f"{API}/molecular-profiles/{PROFILE}/molecular-data/fetch",
                      params={"projection": "SUMMARY"},
                      json={"entrezGeneIds": [int(i) for i in entrez_ids], "sampleListId": SAMPLE_LIST},
                      timeout=120)
    r.raise_for_status()
    rec = pd.DataFrame(r.json())
    if rec.empty:
        return pd.DataFrame()
    rec["value"] = pd.to_numeric(rec["value"], errors="coerce")
    return rec.pivot_table(index="sampleId", columns="entrezGeneId", values="value", aggfunc="first")

def to_patient_index(df):
    out = df.copy()
    out.index = [i[:12] for i in out.index]
    return out[~out.index.duplicated(keep="first")]

# --- جيناتنا الأربعة ---
em = entrez_map(OUR_GENES)
inv = {v: k for k, v in em.items()}
expr_p = to_patient_index(fetch_molecular(list(em.values())).rename(columns=inv))
print("جيناتنا:", expr_p.shape)

# --- البقاء + العمر ---
r = requests.get(f"{API}/studies/{STUDY}/clinical-data",
                 params={"clinicalDataType": "PATIENT", "projection": "SUMMARY"}, timeout=60)
r.raise_for_status()
cl = pd.DataFrame(r.json())
cl = cl[cl["clinicalAttributeId"].isin(["OS_STATUS", "OS_MONTHS", "AGE"])]
surv_df = cl.pivot_table(index="patientId", columns="clinicalAttributeId", values="value", aggfunc="first")
surv_df["OS_MONTHS"] = pd.to_numeric(surv_df["OS_MONTHS"], errors="coerce")
surv_df["event"] = surv_df["OS_STATUS"].astype(str).str.startswith("1").astype(int)
if "AGE" in surv_df:
    surv_df["AGE"] = pd.to_numeric(surv_df["AGE"], errors="coerce")
    print("العمر متوفر لـ", int(surv_df["AGE"].notna().sum()), "مريض")
else:
    print("⚠️ العمر غير متوفر في هذه الدراسة - سيُتخطى الاختبار A")
print("بيانات البقاء:", surv_df.shape)

# --- مجموعة جينات عشوائية للمقارنة (الاختبار C) ---
def get_all_entrez_ids():
    ids = []
    try:
        for page in range(0, 10):
            rr = requests.get(f"{API}/genes", params={"projection": "SUMMARY", "pageSize": 10000,
                                                      "pageNumber": page}, timeout=60)
            rr.raise_for_status()
            chunk = rr.json()
            if not chunk:
                break
            ids += [g["entrezGeneId"] for g in chunk if g.get("type", "protein-coding") == "protein-coding"]
    except Exception as e:
        print("تعذّر GET /genes، سأستخدم بديلاً:", repr(e)[:80])
    if len(ids) < 2000:
        ids = list(range(1, 100000))
    return ids

rng = np.random.default_rng(2026)
all_ids = np.array(sorted(set(get_all_entrez_ids()) - set(em.values())))
sample_ids = rng.choice(all_ids, size=min(7000, len(all_ids)), replace=False)

pool_parts = []
for i in range(0, len(sample_ids), 400):
    try:
        part = fetch_molecular(sample_ids[i:i + 400])
        if not part.empty:
            pool_parts.append(part)
    except Exception as e:
        print("دفعة فاشلة (تُتجاوز):", repr(e)[:70])
pool = pd.concat(pool_parts, axis=1) if pool_parts else pd.DataFrame()
pool = to_patient_index(pool)
keep = (pool.notna().mean() > 0.95) & (pool.mean() > 20)      # جينات معبَّر عنها فعلاً
pool = pool.loc[:, keep]
print(f"مجموعة الجينات العشوائية: {pool.shape[1]} جيناً صالحاً × {pool.shape[0]} مريض")

## 3) إعادة إنتاج النتيجة الأولى (للتأكد أن كل شيء سليم)

In [ ]:
from lifelines import CoxPHFitter, KaplanMeierFitter

vmem = vmem_from_roles(expr_p[OUR_GENES], ROLES)

def make_data(x, extra=None):
    d = surv_df.join(x.rename("x"), how="inner")
    cols = ["OS_MONTHS", "event", "x"] + (extra or [])
    return d[cols].dropna()

def cox_fit(d):
    c = CoxPHFitter()
    c.fit(d, duration_col="OS_MONTHS", event_col="event")
    return c.summary.loc["x"]

d0 = make_data(vmem)
s0 = cox_fit(d0)
sd = d0["x"].std()
print(f"المرضى: {len(d0)} | وفيات: {int(d0['event'].sum())} | انحراف Vmem المعياري: {sd:.2f} mV")
print(f"الأساس: coef={s0['coef']:.4f}  HR/mV={s0['exp(coef)']:.3f}  p={s0['p']:.4f}")
print(f"        HR لكل انحراف معياري واحد = {np.exp(s0['coef'] * sd):.2f}")

## الاختبار A — هل تبقى العلاقة بعد ضبط العمر؟

In [ ]:
resA = None
if "AGE" in surv_df and surv_df["AGE"].notna().sum() > 50:
    dA = make_data(vmem, extra=["AGE"])
    c = CoxPHFitter()
    c.fit(dA, duration_col="OS_MONTHS", event_col="event")
    resA = c.summary
    print(f"المرضى: {len(dA)}")
    print(resA[["coef", "exp(coef)", "p"]].round(4))
    print()
    print(f"Vmem بعد ضبط العمر: HR/mV = {resA.loc['x','exp(coef)']:.3f}, p = {resA.loc['x','p']:.4f}")
else:
    print("لا توجد بيانات عمر كافية - تم التخطي")

## الاختبار B — هل هو تأثير جين واحد؟

In [ ]:
rows = []
# 1) كل جين وحده (log2، ومعياري)
for g in OUR_GENES:
    x = np.log2(expr_p[g].clip(lower=0) + 1)
    x = (x - x.mean()) / x.std()
    s = cox_fit(make_data(x))
    rows.append(dict(test=f"جين وحده: {g}", HR=np.exp(s["coef"]), p=s["p"]))

# 2) Vmem بعد حذف جين واحد في كل مرة
for g in OUR_GENES:
    roles2 = {ch: [x for x in gs if x != g] for ch, gs in ROLES.items()}
    v2 = vmem_from_roles(expr_p[[x for x in OUR_GENES if x != g]], roles2)
    s = cox_fit(make_data(v2))
    rows.append(dict(test=f"Vmem بدون {g}", HR=np.exp(s["coef"]), p=s["p"]))

tabB = pd.DataFrame(rows)
tabB["HR"] = tabB["HR"].round(3); tabB["p"] = tabB["p"].round(4)
print("(HR لكل جين وحده = لكل انحراف معياري؛ HR لـ Vmem = لكل 1 mV)")
tabB

## الاختبار C — هل تعطي جينات عشوائية نتيجة مماثلة؟ (الأهم)

In [ ]:
import matplotlib.pyplot as plt

N_PERM = 1000
rng = np.random.default_rng(7)
pool_genes = list(pool.columns)
n_na, n_k, n_cl = len(ROLES["Na"]), len(ROLES["K"]), len(ROLES["Cl"])

# محاذاة المرضى مرة واحدة لتسريع الحلقة
common = surv_df.index.intersection(pool.index).intersection(expr_p.index)
base = surv_df.loc[common, ["OS_MONTHS", "event"]].dropna()
common = base.index
pool_c = pool.loc[common]

def z_of(v):
    d = base.copy(); d["x"] = v.loc[common].values
    c = CoxPHFitter(); c.fit(d, duration_col="OS_MONTHS", event_col="event")
    return float(c.summary.loc["x", "z"])

z_obs = z_of(vmem)
z_null = []
for i in range(N_PERM):
    g = rng.choice(pool_genes, size=n_na + n_k + n_cl, replace=False)
    roles_r = {"Na": list(g[:n_na]), "K": list(g[n_na:n_na + n_k]), "Cl": list(g[n_na + n_k:])}
    try:
        z_null.append(z_of(vmem_from_roles(pool_c[list(g)], roles_r)))
    except Exception:
        continue
z_null = np.array(z_null)

p_two = (1 + np.sum(np.abs(z_null) >= abs(z_obs))) / (1 + len(z_null))
p_one = (1 + np.sum(z_null >= z_obs)) / (1 + len(z_null))
print(f"قيمة z لجيناتنا: {z_obs:.2f}")
print(f"عدد المحاكاة الناجحة: {len(z_null)}")
print(f"p تجريبي (الاتجاهان): {p_two:.4f}   |   p تجريبي (اتجاه فرضية ليفين فقط): {p_one:.4f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(z_null, bins=40, color="#9ecae1", edgecolor="white")
ax.axvline(z_obs, color="crimson", lw=2, label=f"Our channel genes (z={z_obs:.2f})")
ax.set_xlabel("Cox z-score of predicted Vmem"); ax.set_ylabel("Number of random gene sets")
ax.set_title(f"Random gene sets vs ours (empirical p = {p_two:.3f})")
ax.legend(); plt.tight_layout(); plt.savefig("null_distribution.png", dpi=150); plt.show()

## الاختبار D — الخلاصة بكلمات بسيطة

In [ ]:
print("=" * 50)
print("الخلاصة")
print("=" * 50)

# A
if resA is not None:
    pA = resA.loc["x", "p"]
    print(("✅" if pA < 0.05 else "⚠️"), f"A (العمر): بعد ضبط العمر p = {pA:.4f}",
          "→ العلاقة صامدة" if pA < 0.05 else "→ العلاقة ضعفت، والعمر يفسّر جزءاً منها")
else:
    print("⬜ A (العمر): لم يُجرَ")

# B
loo = tabB[tabB["test"].str.startswith("Vmem بدون")]
n_ok = int((loo["p"] < 0.05).sum())
print(("✅" if n_ok == len(loo) else "⚠️"),
      f"B (جين واحد): العلاقة بقيت دالة عند حذف {n_ok} من أصل {len(loo)} جينات")
bad = loo[loo["p"] >= 0.05]["test"].tolist()
if bad:
    print("    ⚠️ تضعف النتيجة عند:", "، ".join(bad), "← هذا الجين يحمل معظم الإشارة")

# C
print(("✅" if p_two < 0.05 else "⚠️"), f"C (جينات عشوائية): p تجريبي = {p_two:.4f}",
      "→ جينات القنوات أفضل من العشوائية" if p_two < 0.05
      else "→ جينات عشوائية تعطي نتيجة مماثلة؛ الإشارة ليست خاصة بالقنوات")

print()
ok = (resA is None or resA.loc["x", "p"] < 0.05) and n_ok == len(loo) and p_two < 0.05
if ok:
    print("🟢 النتيجة صمدت أمام الاختبارات الثلاثة: تستحق الانتقال لخطوة التحقق على مجموعة بيانات مستقلة.")
else:
    print("🟡 النتيجة لم تصمد في كل الاختبارات. هذا ليس فشلاً، بل معلومة تحدد أين نذهب بعد ذلك.")

## حدود يجب ذكرها دائماً

- mRNA ليس نشاط القناة الفعلي.
- المرجع هو متوسط الأورام نفسها وليس نسيجاً سليماً (GTEx خطوة لاحقة).
- أربعة جينات فقط.
- حتى لو صمدت النتيجة هنا، يلزم التحقق على مجموعة بيانات مستقلة (مثل CGGA أو Rembrandt) قبل أي ادعاء.